In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# An Interpreter for a Simple Programming Language

In this notebook we develop an interpreter for a small programming language.  We will utilize the F# port of the parser generator
`Larkita` to define our language rules.  The parser generated by `Larkita` returns the abstract syntax tree of a
program as a nested tuple (in F#, a value of the type `Ast`, see `01-Calculator.ipynb`), so we do not have to
convert a parse tree.  Below, the tuples are converted into a typed abstract syntax tree.

In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open LarkitaAst

## Specification of the Grammar

Below is the grammar for our simple `C`-like language, written in *Extended Backus-Naur Form* (EBNF).  It is stored
in the file `sl.lark`.

Our programming language allows both single-line (`//`) and multi-line (`/* */`) comments.  We instruct the parser
to entirely `%ignore` them alongside standard whitespace.

The grammar also determines the abstract syntax tree:  The string literals of an alternative form the *head* of the
tuple that is returned for this alternative, while all other symbols become its *children*.

* In contrast to `C`, the body of an `if` statement or a `while` loop always has to be enclosed in braces, even if
  it consists of a single statement.  Therefore, the body is a `program`, i.e. a sequence of statements.
* The directive `%ignore: "(", ")", "{", "}", ";"` (note the colon) removes the parentheses, the braces, and the
  semicolon from the heads of *all* alternatives, since these literals carry no information.  Hence the alternative
    ```
       "while" "(" bool_expr ")" "{" program "}"
    ```
  yields the tuple
  ```
       ('while', test, body)
  ```
  where `body` is the tuple
  `('program', stmnt, ...)` of the statements in braces, and the alternative
  `ID ":=" expr ";"` yields `(':=', ('ID', name), expr)`.
* An *inner alternative* like `("+" | "-")` abbreviates several alternatives:  The alternative
  `expr ("+" | "-") product` stands for the two alternatives `expr "+" product` and `expr "-" product`.  Hence the
  operator that has been read is the head, e.g. `x - 1` yields `('-', ('ID', 'x'), ('INT', '1'))`.
* An alternative that consists of a single symbol passes the abstract syntax tree of this symbol through.  Hence a
  parenthesized expression is the expression inside.
* An *alias* `-> name` replaces the head.  Without the alias, the alternative `expr ";"` would consist of a single
  symbol and would therefore return the expression itself.  With the alias, it yields `('expr', expr)`.
* The arguments of a function call are written inline as `[expr ("," expr)*]`.  The children of an optional part and
  of a repetition are spliced into the tuple of the alternative, so a function call yields
  `('call', ('ID', name), arg_1, ..., arg_n)`.
* The program is the first rule of the grammar and therefore its start symbol.  As the alternative `stmnt*` contains
  no literal, the name of the rule is used as head:  A program yields the tuple `('program', stmnt_1, ..., stmnt_n)`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "sl.lark")

The typed AST distinguishes numeric literals, variable names, expressions and statements.  The same types also
describe the extensions used in the exercise notebook `Interpreter-Frame.ipynb`.

In [ ]:
open System.Numerics

type Value = Integer of BigInteger | Real of float | Text of string | Boolean of bool
type BinaryOp = Add | Subtract | Multiply | Divide | Modulo | Equal | NotEqual | Less | Greater | LessEqual | GreaterEqual | And | Or
type Expr =
    | Literal of Value
    | Variable of string
    | Binary of BinaryOp * Expr * Expr
    | Not of Expr
    | Negative of Expr
    | Call of string * Expr list

type Statement =
    | Block of Statement list
    | Assign of string * Expr
    | Expression of Expr
    | If of Expr * Statement
    | While of Expr * Statement
    | For of Statement * Expr * Statement * Statement
    | Define of string * string list * Statement
    | Return of Expr

## The Abstract Syntax Tree

The parser returns the abstract syntax tree as a nested tuple.  This tuple has exactly the structure that is needed
by our evaluation and visualization functions.  The shapes that are produced are the following:

| construct              | nested tuple                                       |
|:-----------------------|:---------------------------------------------------|
| program, body          | `('program', stmnt, ...)`                          |
| assignment             | `(':=', ('ID', name), expr)`                       |
| expression statement   | `('expr', expr)`                                   |
| conditional, loop      | `('if', test, body)`, `('while', test, body)`      |
| binary operator        | `(op, lhs, rhs)`, for example `('+', lhs, rhs)`    |
| number                 | `('INT', digits)`                                  |
| variable               | `('ID', name)`                                     |
| function call          | `('call', ('ID', name), arg, ...)`                 |

The functions `buildExpr` and `buildStatement` convert these tuples into the typed AST:  for example, the tuple
`('while', test, body)` becomes `While(test, body)`.  `slParser` is the parser generated from `sl.lark`; its
states are described in the file `sl.out`.

In [ ]:
let binaryOps =
    Map.ofList [ "+", Add; "-", Subtract; "*", Multiply; "/", Divide; "%", Modulo
                 "==", Equal; "!=", NotEqual; "<", Less; ">", Greater; "<=", LessEqual; ">=", GreaterEqual
                 "&&", And; "||", Or ]

let unexpected (t: Ast) = failwithf "Unexpected tuple in the abstract syntax tree: %s" (toString t)

let rec buildExpr (t: Ast) : Expr =
    match t with
    | Tok("INT", digits) -> Literal(Integer(BigInteger.Parse(digits.Replace("_", ""), System.Globalization.CultureInfo.InvariantCulture)))
    | Tok("FLOAT", digits) -> Literal(Real(System.Double.Parse(digits.Replace("_", ""), System.Globalization.CultureInfo.InvariantCulture)))
    | Tok("STRING", text) -> Literal(Text(text.Substring(1, text.Length - 2)))
    | Tok("ID", name) -> Variable name
    | Node([ "call" ], Tok("ID", name) :: arguments) -> Call(name, List.map buildExpr arguments)
    | Node([ "!" ], [ argument ]) -> Not(buildExpr argument)
    | Node([ op ], [ left; right ]) when binaryOps.ContainsKey op -> Binary(binaryOps[op], buildExpr left, buildExpr right)
    | _ -> unexpected t

let rec buildStatement (t: Ast) : Statement =
    match t with
    | Node([ "program" ], statements) -> Block(List.map buildStatement statements)
    | Node([ ":=" ], [ Tok("ID", name); expr ]) -> Assign(name, buildExpr expr)
    | Node([ "expr" ], [ expr ]) -> Expression(buildExpr expr)
    | Node([ "if" ], [ test; body ]) -> If(buildExpr test, buildStatement body)
    | Node([ "while" ], [ test; body ]) -> While(buildExpr test, buildStatement body)
    | Node([ "for" ], [ initial; test; update; body ]) ->
        For(buildStatement initial, buildExpr test, buildStatement update, buildStatement body)
    | Node([ "function" ], [ Tok("ID", name); Node([ "parameters" ], parameters); body ]) ->
        let parameter = function Tok("ID", p) -> p | t -> unexpected t
        Define(name, List.map parameter parameters, buildStatement body)
    | Node([ "return" ], [ expr ]) -> Return(buildExpr expr)
    | _ -> unexpected t

/// the typed statement of a program and the tokens of a text, for the parser generated from a grammar
let programWith (p: Parser) (source: string) = buildStatement (parse p source)
let tokensWith (p: Parser) (source: string) = lex p source |> List.map (fun t -> t.Type, t.Value, t.Line, t.Column)

// The grammar is found next to this notebook, also when the exported module is loaded from elsewhere.
let slParser = Larkita.Open(System.IO.Path.Combine(__SOURCE_DIRECTORY__, "sl.lark"), outFile = System.IO.Path.Combine(__SOURCE_DIRECTORY__, "sl.out"))

module Parse =
    let program source = programWith slParser source
    let file path = program (System.IO.File.ReadAllText path)
    let tokens source = tokensWith slParser source

For example, the statement `x := 1 + 2 * y;` yields the following tuple:

In [ ]:
let tree = parse slParser "x := 1 + 2 * y;"
printfn "%s" (toString tree)
checkNotebook (Parse.program "x := 1 + 2 * y;" = Block [ Assign("x", Binary(Add, Literal(Integer 1I), Binary(Multiply, Literal(Integer 2I), Variable "y"))) ])

The lexer keeps track of lines and columns.  `Parse.tokens` shows the tokens of a program together with their positions.

In [ ]:
let testScanner file =
    let source = System.IO.File.ReadAllText file
    printfn "%s" source
    printfn "%A" (Parse.tokens source)

In [ ]:
testScanner "sum.sl"

In [ ]:
testScanner "factorial.sl"

An example program that conforms to this grammar is stored in the file `sum.sl`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "sum.sl")

A program is a list of statements.

`Parse.program` returns a typed `Statement` tree; a conversion creates the Graphviz representation.

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

let opName = function
    | Add -> "+" | Subtract -> "-" | Multiply -> "*" | Divide -> "/" | Modulo -> "%"
    | Equal -> "==" | NotEqual -> "!=" | Less -> "<" | Greater -> ">" | LessEqual -> "<=" | GreaterEqual -> ">="
    | And -> "&&" | Or -> "||"
let valueText = function
    | Integer n -> string n
    | Real n -> n.ToString("G17", System.Globalization.CultureInfo.InvariantCulture)
    | Text s -> s
    | Boolean b -> string b
let rec exprTree = function
    | Literal value -> Leaf(valueText value)
    | Variable name -> Leaf name
    | Binary(op, left, right) -> Node(opName op, [exprTree left; exprTree right])
    | Not expr -> Node("!", [exprTree expr])
    | Negative expr -> Node("-", [exprTree expr])
    | Call(name, args) -> Node("call " + name, List.map exprTree args)
let rec statementTree = function
    | Block statements -> Node("block", List.map statementTree statements)
    | Assign(name, expr) -> Node(":=", [Leaf name; exprTree expr])
    | Expression expr -> Node("expression", [exprTree expr])
    | If(test, body) -> Node("if", [exprTree test; statementTree body])
    | While(test, body) -> Node("while", [exprTree test; statementTree body])
    | For(init, test, update, body) -> Node("for", [statementTree init; exprTree test; statementTree update; statementTree body])
    | Define(name, parameters, body) -> Node("function " + name, [Node("parameters", List.map Leaf parameters); statementTree body])
    | Return expr -> Node("return", [exprTree expr])
let parse file =
    let ast = Parse.file file
    printfn "%s" (treeToDot (statementTree ast))
    ast

`parse file` reads the program, prints its DOT tree.

In [ ]:
parse "sum.sl" |> ignore

F# infers the mutually recursive types from their `let rec` group; forward declarations are unnecessary.

Executing a block folds over its statements. Each statement receives the current variable map and returns the next map.

`execute io values statement` implements assignment, expression evaluation, conditionals and loops. The explicit IO record makes examples reproducible while preserving an interactive `main file` entry point.

`evaluateBool` evaluates a comparison from the same variable environment.

`evaluate` handles numeric literals, variables, arithmetic and the `read`/`print` built-ins. The variable map and IO record retain `Value`: integer addition, subtraction, multiplication and modulo stay exact, while `/` produces a `Real`, as in the original Python interpreter. `read` parses an integer and `print` formats it without conversion to `float`.

### Numeric values and comparisons

The exercise interpreter uses the same `Value` domain. These shared helpers compare integers directly and compare a mixed integer/real pair against the real value actually represented in memory. The floor quotient gives modulo the sign of its divisor, including negative operands. The `Text` and `Boolean` comparison cases are used by the exercise extensions.

In [ ]:
open System.Numerics

let number = function
    | (Integer _ | Real _) as value -> value
    | value -> failwithf "Expected a number, got %A" value
let asNumber = function
    | Integer n -> float n
    | Real n -> n
    | value -> failwithf "Expected a number, got %A" value

// Compare the integer with the actual represented real, without rounding it first.
let compareIntegerReal (integer: BigInteger) (real: float) =
    if System.Double.IsNaN real then failwith "NaN cannot be ordered"
    elif System.Double.IsPositiveInfinity real then -1
    elif System.Double.IsNegativeInfinity real then 1
    else
        let whole = BigInteger real
        let result = compare integer whole
        if result <> 0 then result else compare (float integer) real
let compareValues left right =
    match left, right with
    | Integer a, Integer b -> compare a b
    | Integer a, Real b -> compareIntegerReal a b
    | Real a, Integer b -> -(compareIntegerReal b a)
    | Real a, Real b -> compare a b
    | Text a, Text b -> System.String.CompareOrdinal(a, b)
    | Boolean a, Boolean b -> compare a b
    | _ -> failwithf "Cannot compare %A and %A" left right
let equalValues left right =
    match left, right with
    | Real a, Real b -> a = b
    | Integer _, Real b when System.Double.IsNaN b -> false
    | Real a, Integer _ when System.Double.IsNaN a -> false
    | (Integer _ | Real _), (Integer _ | Real _) -> compareValues left right = 0
    | _ -> left = right

// .NET integer division truncates; Python's integer modulo uses a floor quotient.
let floorDivide (left: BigInteger) (right: BigInteger) =
    if right.IsZero then failwith "Division by zero"
    let quotient, remainder = left / right, left % right
    if not remainder.IsZero && left.Sign <> right.Sign then quotient - BigInteger.One else quotient

// Python modulo has the divisor's sign, including an exact zero remainder.
let floatModulo (left: float) (right: float) =
    if right = 0.0 then failwith "Modulo by zero"
    let remainder = left % right
    if remainder = 0.0 then System.Math.CopySign(0.0, right)
    elif (remainder < 0.0) <> (right < 0.0) then remainder + right
    else remainder


Division returns a floating-point value even for two integers. To preserve Python semantics for large operands, `divideIntegers` rounds the exact ratio to the nearest binary double, with ties going to the even significand. This also handles ratios of integers that individually exceed the floating-point range. Mixed real arithmetic uses the ordinary .NET floating-point operations. Real modulo starts with the native remainder, then applies Python’s divisor-sign rule; this avoids forming an overflowing quotient and preserves signed zero.

In [ ]:
// Round the exact ratio once, rather than rounding each large operand to float.
let divideIntegers (numerator: BigInteger) (denominator: BigInteger) =
    if denominator.IsZero then failwith "Division by zero"
    let sign = if (numerator.Sign < 0) <> (denominator.Sign < 0) then -1.0 else 1.0
    let a, b = BigInteger.Abs numerator, BigInteger.Abs denominator
    if a.IsZero then sign * 0.0
    else
        let candidate = a.GetBitLength() - b.GetBitLength()
        if candidate > 1024L then failwith "Integer division result is too large for a real value"
        elif candidate < -1075L then sign * 0.0
        else
            let power = int candidate
            let belowPower =
                if power >= 0 then a < (b <<< power)
                else (a <<< -power) < b
            let exponent = if belowPower then power - 1 else power
            // Normal doubles have 53 significant bits; subnormals use units of 2^-1074.
            let scale = if exponent < -1022 then 1074 else 52 - exponent
            let scaledA, scaledB =
                if scale >= 0 then a <<< scale, b
                else a, b <<< -scale
            let quotient, remainder = scaledA / scaledB, scaledA % scaledB
            let twiceRemainder = remainder * 2I
            let rounded =
                if twiceRemainder > scaledB || (twiceRemainder = scaledB && not quotient.IsEven)
                then quotient + 1I else quotient
            let result = System.Math.ScaleB(float rounded, -scale)
            if System.Double.IsInfinity result then failwith "Integer division result is too large for a real value"
            sign * result

let arithmetic op left right =
    match left, right with
    | Integer a, Integer b ->
        match op with
        | Add -> Integer(a + b) | Subtract -> Integer(a - b) | Multiply -> Integer(a * b)
        | Divide -> Real(divideIntegers a b)
        | Modulo -> Integer(a - floorDivide a b * b)
        | _ -> failwith "Expected an arithmetic operator"
    | _ ->
        let a, b = asNumber left, asNumber right
        match op with
        | Add -> Real(a + b) | Subtract -> Real(a - b) | Multiply -> Real(a * b)
        | Divide -> Real(divide a b) | Modulo -> Real(floatModulo a b)
        | _ -> failwith "Expected an arithmetic operator"


In [ ]:
type IO = { Read: unit -> Value; Write: Value -> unit }

let rec evaluate io values = function
    | Literal value -> number value
    | Variable name ->
        match Map.tryFind name values with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Call("read", []) -> number (io.Read())
    | Call("print", [expr]) -> io.Write(evaluate io values expr); Integer 0I
    | Negative expr ->
        match evaluate io values expr with
        | Integer n -> Integer(-n) | Real n -> Real(-n)
        | _ -> failwith "Unary minus expects a number"
    | Binary(op, left, right) ->
        arithmetic op (evaluate io values left) (evaluate io values right)
    | _ -> failwith "Expression is outside the simple language"

let evaluateBool io values = function
    | Binary(op, left, right) ->
        let l, r = evaluate io values left, evaluate io values right
        match op with
        | Equal -> equalValues l r | NotEqual -> not (equalValues l r)
        | Less -> compareValues l r < 0 | Greater -> compareValues l r > 0
        | LessEqual -> compareValues l r <= 0 | GreaterEqual -> compareValues l r >= 0
        | _ -> failwith "Expected a comparison"
    | _ -> failwith "Expected a comparison"

let rec execute io values = function
    | Block statements -> List.fold (execute io) values statements
    | Assign(name, expr) -> Map.add name (evaluate io values expr) values
    | Expression expr -> evaluate io values expr |> ignore; values
    | If(test, body) -> if evaluateBool io values test then execute io values body else values
    | While(test, body) ->
        let rec repeat current =
            if evaluateBool io current test then repeat (execute io current body) else current
        repeat values
    | _ -> failwith "Statement is outside the simple language"

let run io source = execute io Map.empty (Parse.program source)
let main file =
    let read () =
        printf "Please enter a natural number: "
        Integer(BigInteger.Parse(System.Console.ReadLine(), System.Globalization.CultureInfo.InvariantCulture))
    let io = { Read = read; Write = (valueText >> printfn "%s") }
    run io (System.IO.File.ReadAllText file)


In [ ]:
printfn "%s" (System.IO.File.ReadAllText "sum.sl")

In [ ]:
let output = ResizeArray<Value>()
let demoIO = { Read = (fun () -> Integer 10I); Write = output.Add }
let values = run demoIO (System.IO.File.ReadAllText "sum.sl")
checkNotebook (List.ofSeq output = [Integer 5050I])
printfn "%A" values

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "factorial.sl")

In [ ]:
output.Clear()
run demoIO (System.IO.File.ReadAllText "factorial.sl") |> ignore
checkNotebook (List.ofSeq output = [Integer 3628800I])
printfn "%A" output

### Exact integers and Python arithmetic

The checks below would fail if the interpreter rounded every literal or input to `float`.

In [ ]:
output.Clear()
let exactInput = { demoIO with Read = (fun () -> Integer 9007199254740993I) }
let exact = run exactInput "x := read(); y := x + 1; print(y);"
checkNotebook (exact["x"] = Integer 9007199254740993I)
checkNotebook (List.ofSeq output = [Integer 9007199254740994I])
let numeric = run demoIO "q := 3 / 2; r := (0 - 7) % 3; s := 7 % (0 - 3); rounded := 9007199254740993 / 3;"
checkNotebook (numeric["q"] = Real 1.5)
checkNotebook (numeric["r"] = Integer 2I && numeric["s"] = Integer -2I)
checkNotebook (numeric["rounded"] = Real 3002399751580331.0)
let compared = run demoIO "x := 0; if (9007199254740993 > 9007199254740992) { x := 1; }"
checkNotebook (compared["x"] = Integer 1I)
printfn "%s" (valueText exact["y"])

### Rendered program tree

The diagram makes the nesting and operator structure visible in Jupyter.

In [ ]:
let svg = statementTree (Parse.file "sum.sl") |> treeToDot |> renderDot
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(svg), "text/html") |> ignore
